# HuBMAP Vasculature — Dual-Stream Inference (Kaggle)

**YOLO Fast Stream** (4-fold WBF) + **MedSAM-2 Heavy Stream** (LoRA-FT, bbox-prompted refinement)

依賴：上傳一個 dataset 包含 weights + code + vendored MedSAM2 (`package_for_kaggle.sh` 產出)。
改下面 `DATASET_NAME` 成你的 dataset slug 後段名稱。

In [ ]:
# ===== Cell 1: 路徑設定 =====
import os, shutil, subprocess, sys, time
from pathlib import Path

DATASET_NAME = "hubmap-dual-stream-weights"
COMP_NAME = "hubmap-hacking-the-human-vasculature"

DATASET_DIR = Path(f"/kaggle/input/{DATASET_NAME}")
COMP_DIR = Path(f"/kaggle/input/{COMP_NAME}")
WORK = Path("/kaggle/working")

assert DATASET_DIR.exists(), f"找不到 {DATASET_DIR}，請確認 dataset 已 attach"
assert COMP_DIR.exists(), f"找不到比賽資料 {COMP_DIR}"

print("Dataset 內容：")
for p in sorted(DATASET_DIR.iterdir()):
    print(" ", p.name)

In [ ]:
# ===== Cell 2: 連結 vendored MedSAM2 到 WORK/third_party/MedSAM2 =====
third_party_root = WORK / "third_party"
third_party_root.mkdir(parents=True, exist_ok=True)
dst = third_party_root / "MedSAM2"

# 找到含 sam2/__init__.py 的那層
candidates = [
    DATASET_DIR / "third_party_MedSAM2" / "MedSAM2",  # 情況 B
    DATASET_DIR / "third_party_MedSAM2",              # 情況 A
    DATASET_DIR / "MedSAM2",                          # 萬一變這樣
]
src = next((c for c in candidates if (c / "sam2" / "__init__.py").exists()), None)
assert src is not None, f"找不到 MedSAM2 source，candidates={[str(c) for c in candidates]}"

if dst.exists() or dst.is_symlink():
    dst.unlink() if dst.is_symlink() else shutil.rmtree(dst)
dst.symlink_to(src)
print(f"symlink: {dst} -> {src}")
print("sam2 內容前 5 個：", sorted(os.listdir(src / "sam2"))[:5])


In [ ]:
# ===== Cell 3: 複製推理程式碼到 working =====
for name in ("medsam2_stream.py", "predict_dual_stream.py", "predict_ensemble.py",
             "train.py", "dataprocess.py"):
    src = DATASET_DIR / "code" / name
    dst = WORK / name
    shutil.copy(src, dst)
    print(f"  {dst}")
# 驗證 medsam2_stream.py 看得到 third_party/MedSAM2
assert (WORK / "third_party" / "MedSAM2" / "sam2").exists(), "third_party 不在 WORK 旁邊！"

In [ ]:
# ===== Cell 4: 離線安裝缺少的套件（從 dataset/wheels/）=====
WHEELS = DATASET_DIR / "wheels"
assert WHEELS.exists() and any(WHEELS.glob("*.whl")), f"找不到 {WHEELS}，請確認 package_for_kaggle.sh 有產 wheels/"

!pip install --no-index --find-links={WHEELS} \
    peft safetensors hydra-core iopath \
    omegaconf antlr4-python3-runtime portalocker \
    ultralytics --no-deps -q

# 驗證
import importlib
for pkg in ("peft", "hydra", "iopath", "safetensors", "omegaconf", "ultralytics"):
    try:
        m = importlib.import_module(pkg)
        print(f"  ✓ {pkg} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"  ✗ {pkg}: {e}")

In [ ]:
# ===== Cell 5: 組推理指令 =====
os.chdir(WORK)

yolo_weights = sorted((DATASET_DIR / "yolo").glob("fold*.pt"))
medsam_base = DATASET_DIR / "medsam2" / "base" / "MedSAM2_latest.pt"
lora_dirs = sorted((DATASET_DIR / "medsam2").glob("fold*/lora_best"))
decoder_pts = sorted((DATASET_DIR / "medsam2").glob("fold*/decoder_best.pt"))

print("YOLO weights:", [p.name for p in yolo_weights])
print("MedSAM base :", medsam_base.name, "exists:", medsam_base.exists())
print("LoRA dirs   :", [p.parent.name + "/" + p.name for p in lora_dirs])
print("Decoder pts :", [p.parent.name + "/" + p.name for p in decoder_pts])

In [ ]:
# ===== Cell 6: 跑雙串流推理 =====
out_dir = WORK / "preds"
out_dir.mkdir(exist_ok=True)

cmd = [
    sys.executable, "predict_dual_stream.py",
    "--yolo-weights", *[str(p) for p in yolo_weights],
    "--medsam-ckpt", str(medsam_base),
    "--medsam-cfg", "configs/sam2.1_hiera_t512.yaml",
    "--medsam-loras", *[str(p) for p in lora_dirs],
    "--medsam-decoders", *[str(p) for p in decoder_pts],
    "--src", str(COMP_DIR / "test"),
    "--meta", str(COMP_DIR / "tile_meta.csv"),
    "--out", str(out_dir),
    "--conf-high", "0.25",
    "--wbf-iou", "0.55",
    "--device", "cuda",
]
print(" ".join(cmd))
t0 = time.time()
subprocess.run(cmd, check=True, cwd=str(WORK))
print(f"\n推理用時 {time.time()-t0:.1f}s")

In [ ]:
# ===== Cell 7: 產 submission.csv =====
subprocess.run([
    sys.executable, "train.py", "submit",
    "--preds", str(out_dir),
    "--out", str(WORK / "submission.csv"),
], check=True, cwd=str(WORK))

In [ ]:
# ===== Cell 8: 檢查 submission =====
import pandas as pd
df = pd.read_csv(WORK / "submission.csv")
print(df.shape)
print("非空 prediction：", (df["prediction_string"].fillna("") != "").sum())
df.head(3)